# SFT Tokenization Walkthrough

This notebook uses the same YAML config, tokenizer, processor, template, and multimodal expansion path as training, and focuses on how one video example turns into prompt tokens and supervised target tokens.

In [ ]:
from pathlib import Path

CONFIG_RELATIVE_PATH = Path('SFT/examples/local/robovqa_cosmos_sft_qwen3_vl_4b_full_q1.yaml')
EXAMPLE_INDEX = 0
PROMPT_PREVIEW_CHARS = 4000
FRAME_PREVIEW_COUNT = 6
VIDEO_WIDTH = 720


In [ ]:
import base64
import importlib
import math
import sys
from io import BytesIO
from pathlib import Path

import av
import numpy as np
import pandas as pd
import yaml
from IPython.display import HTML, Markdown, display
from PIL import Image, ImageOps, ImageDraw


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'SFT').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate verl-post-training repo root from current working directory.')


def load_video_info(video_path: Path) -> dict[str, float | int]:
    container = av.open(str(video_path), 'r')
    stream = next(stream for stream in container.streams if stream.type == 'video')
    info = {
        'width': int(stream.width or 0),
        'height': int(stream.height or 0),
        'fps': float(stream.average_rate) if stream.average_rate is not None else 0.0,
        'frame_count': int(stream.frames or 0),
    }
    container.close()
    return info


def display_embedded_video(video_path: Path, width: int = 720) -> None:
    mime = 'video/mp4'
    encoded = base64.b64encode(video_path.read_bytes()).decode('ascii')
    html = f'''<video controls width="{width}" style="max-width: 100%; height: auto;">
    <source src="data:{mime};base64,{encoded}" type="{mime}">
    Your browser does not support embedded video.
</video>'''
    display(HTML(html))


def sample_video_frames(video_path: Path, count: int = 6) -> list[Image.Image]:
    container = av.open(str(video_path), 'r')
    stream = next(stream for stream in container.streams if stream.type == 'video')
    total = int(stream.frames or 0)
    if total <= 0:
        container.close()
        return []
    indices = np.linspace(0, max(total - 1, 0), min(count, total)).astype(int)
    wanted = set(indices.tolist())
    rendered = []
    for frame_idx, frame in enumerate(container.decode(stream)):
        if frame_idx not in wanted:
            continue
        img = frame.to_image().convert('RGB')
        canvas = ImageOps.pad(img, (220, 160), color=(245, 245, 245))
        draw = ImageDraw.Draw(canvas)
        draw.rectangle((0, 0, 90, 22), fill=(0, 0, 0))
        draw.text((6, 4), f'frame {frame_idx}', fill=(255, 255, 255))
        rendered.append(canvas)
        if len(rendered) == len(wanted):
            break
    container.close()
    return rendered


def show_frame_strip(images: list[Image.Image]) -> None:
    if not images:
        display(Markdown('_No frame preview available._'))
        return
    width = sum(img.width for img in images)
    height = max(img.height for img in images)
    strip = Image.new('RGB', (width, height), color=(255, 255, 255))
    x = 0
    for img in images:
        strip.paste(img, (x, 0))
        x += img.width
    buffer = BytesIO()
    strip.save(buffer, format='PNG')
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
    display(HTML(f'<img src="data:image/png;base64,{encoded}" style="max-width: 100%; height: auto;" />'))


ROOT_DIR = find_repo_root(Path.cwd())
CONFIG_PATH = ROOT_DIR / CONFIG_RELATIVE_PATH
SCRIPTS_DIR = ROOT_DIR / 'SFT' / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import tokenization_inspector
importlib.reload(tokenization_inspector)
from tokenization_inspector import analyze_example, build_mask_segments, build_modality_breakdown

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 200)


In [ ]:
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    raw_yaml = yaml.safe_load(handle)

analysis = analyze_example(CONFIG_PATH, EXAMPLE_INDEX)
summary = analysis['summary']
video_path = Path(summary['video_paths'][0]) if summary['video_paths'] else None
video_info = load_video_info(video_path) if video_path is not None else None

raw_yaml_tokenization = {
    key: raw_yaml.get(key)
    for key in [
        'model_name_or_path',
        'template',
        'cutoff_len',
        'image_max_pixels',
        'video_max_pixels',
        'video_min_pixels',
        'learning_rate',
        'per_device_train_batch_size',
        'gradient_accumulation_steps',
        'bf16',
        'torch_compile',
    ]
}

display(Markdown('## Config Loaded From YAML'))
display(pd.DataFrame([raw_yaml_tokenization]).T.rename(columns={0: 'yaml_value'}))

display(Markdown('## Resolved Runtime Values Used By The Actual Pipeline'))
display(pd.DataFrame([analysis['resolved_config']]).T.rename(columns={0: 'resolved_value'}))

display(Markdown(
    f"## Dataset Row\n- file: `{analysis['dataset_file']}`\n- index: `{EXAMPLE_INDEX}`"
))


In [ ]:
if video_path is None:
    display(Markdown('No video attached to this example.'))
else:
    display(Markdown(f"## Video\n`{video_path}`"))
    display_embedded_video(video_path, width=VIDEO_WIDTH)
    display(Markdown('### Frame Preview'))
    show_frame_strip(sample_video_frames(video_path, count=FRAME_PREVIEW_COUNT))


In [ ]:
display(Markdown('## Raw User Prompt'))
display(Markdown(f"```text\n{analysis['raw_prompt_text']}\n```"))

display(Markdown('## Raw Assistant Target'))
display(Markdown(f"```text\n{analysis['assistant_text']}\n```"))


In [ ]:
display(Markdown('## Step 1: Original Video Geometry'))
if video_info is None:
    display(Markdown('_No video geometry available._'))
else:
    duration = video_info['frame_count'] / video_info['fps'] if video_info['fps'] else None
    original_pixels = video_info['width'] * video_info['height']
    display(pd.DataFrame([
        {'field': 'original_width', 'value': video_info['width']},
        {'field': 'original_height', 'value': video_info['height']},
        {'field': 'original_fps', 'value': round(video_info['fps'], 4)},
        {'field': 'original_frame_count', 'value': video_info['frame_count']},
        {'field': 'original_duration_seconds', 'value': round(duration, 4) if duration is not None else None},
        {'field': 'original_pixels_per_frame', 'value': original_pixels},
        {'field': 'yaml_video_max_pixels', 'value': raw_yaml.get('video_max_pixels')},
    ]))


In [ ]:
display(Markdown('## Step 2: Sampling And Temporal Grouping'))
requested_fps = analysis['resolved_config']['video_fps']
duration = summary['video_duration_seconds']
expected_raw_samples = math.floor(duration * requested_fps) if duration is not None else None
sampling_rows = [
    {'field': 'requested_video_fps', 'value': requested_fps, 'meaning': 'Configured sampling rate from YAML.'},
    {'field': 'video_duration_seconds', 'value': round(duration, 4) if duration is not None else None, 'meaning': 'Duration seen by the processor.'},
    {'field': 'floor(duration * fps)', 'value': expected_raw_samples, 'meaning': 'How many raw frames the sampler asks for before temporal merging.'},
    {'field': 'sampled_raw_frames', 'value': summary['sampled_raw_frames'], 'meaning': 'Raw sampled frames after even-padding if needed.'},
    {'field': 'video_temporal_patch_size', 'value': summary['video_temporal_patch_size'], 'meaning': 'Qwen3-VL groups this many sampled frames into one temporal slice.'},
    {'field': 'final_T', 'value': summary['video_grid_thw'][0], 'meaning': 'Temporal grid size after grouping sampled frames.'},
]
display(pd.DataFrame(sampling_rows))

display(Markdown(
    f"`sampled_raw_frames ({summary['sampled_raw_frames']}) / video_temporal_patch_size ({summary['video_temporal_patch_size']}) = final_T ({summary['video_grid_thw'][0]})`"
))


In [ ]:
display(Markdown('## Step 3: Resizing And Spatial Grid'))
if video_info is None:
    display(Markdown('_No video resize details available._'))
else:
    original_pixels = video_info['width'] * video_info['height']
    max_pixels = raw_yaml.get('video_max_pixels')
    resize_factor = math.sqrt(max_pixels / original_pixels)
    approx_width = video_info['width'] * resize_factor
    approx_height = video_info['height'] * resize_factor
    patch_size = summary['video_patch_size']
    resized_width = summary['resized_frame_width']
    resized_height = summary['resized_frame_height']
    display(pd.DataFrame([
        {'field': 'pixel_budget_resize_factor', 'value': round(resize_factor, 4), 'meaning': 'Initial downscale from original frame pixels to the YAML budget.'},
        {'field': 'approx_resized_width_before_snap', 'value': round(approx_width, 2), 'meaning': 'Width from the pixel budget before model snapping.'},
        {'field': 'approx_resized_height_before_snap', 'value': round(approx_height, 2), 'meaning': 'Height from the pixel budget before model snapping.'},
        {'field': 'video_patch_size', 'value': patch_size, 'meaning': 'Qwen3-VL spatial patch size.'},
        {'field': 'resized_frame_width_after_snap', 'value': resized_width, 'meaning': 'Width after snapping to patch-compatible dimensions.'},
        {'field': 'resized_frame_height_after_snap', 'value': resized_height, 'meaning': 'Height after snapping to patch-compatible dimensions.'},
        {'field': 'grid_H', 'value': summary['video_grid_thw'][1], 'meaning': 'resized_height / patch_size'},
        {'field': 'grid_W', 'value': summary['video_grid_thw'][2], 'meaning': 'resized_width / patch_size'},
    ]))

    display(Markdown(
        f"`grid_H = {resized_height} / {patch_size} = {summary['video_grid_thw'][1]}` and `grid_W = {resized_width} / {patch_size} = {summary['video_grid_thw'][2]}`"
    ))


In [ ]:
display(Markdown('## Step 4: From T, H, W To Video Tokens'))
T, H, W = summary['video_grid_thw']
merge_size = summary['video_merge_size']
merge_area = merge_size * merge_size
cells_per_slice = H * W
soft_per_slice = summary['video_soft_tokens_per_frame']
soft_total = summary['video_soft_tokens_total']

display(pd.DataFrame([
    {'field': 'video_grid_thw', 'value': str(summary['video_grid_thw']), 'meaning': 'Temporal, height, width grid fed to the model.'},
    {'field': 'cells_per_temporal_slice', 'value': cells_per_slice, 'meaning': 'H * W spatial cells in one temporal slice.'},
    {'field': 'merge_size', 'value': merge_size, 'meaning': 'Spatial cells are merged in merge_size x merge_size groups.'},
    {'field': 'merge_area', 'value': merge_area, 'meaning': 'One soft token represents this many spatial cells.'},
    {'field': 'video_soft_tokens_per_frame', 'value': soft_per_slice, 'meaning': '(H * W) / merge_area'},
    {'field': 'video_soft_tokens_total', 'value': soft_total, 'meaning': 'T * video_soft_tokens_per_frame'},
    {'field': 'video_non_soft_overhead_tokens', 'value': summary['video_non_soft_overhead_tokens'], 'meaning': 'Timestamp text plus <|vision_start|>/<|vision_end|> wrappers.'},
    {'field': 'video_added_prompt_tokens', 'value': summary['video_added_prompt_tokens'], 'meaning': 'soft video tokens plus overhead tokens.'},
]))

display(Markdown(
    f"`video_soft_tokens_per_frame = ({H} * {W}) / ({merge_size} * {merge_size}) = {soft_per_slice}`"
))
display(Markdown(
    f"`video_soft_tokens_total = {T} * {soft_per_slice} = {soft_total}`"
))
display(Markdown(
    f"`video_added_prompt_tokens = {soft_total} + {summary['video_non_soft_overhead_tokens']} = {summary['video_added_prompt_tokens']}`"
))


In [ ]:
display(Markdown('## Prompt After Multimodal Expansion'))
display(Markdown(
    'The raw `<video>` placeholder is expanded into timestamp text plus repeated visual placeholder tokens before tokenization.'
))
expanded_preview = analysis['expanded_prompt_text'][:PROMPT_PREVIEW_CHARS]
display(Markdown(f"```text\n{expanded_preview}\n```"))
if len(analysis['expanded_prompt_text']) > PROMPT_PREVIEW_CHARS:
    display(Markdown(f'_Preview truncated at {PROMPT_PREVIEW_CHARS} characters._'))


In [ ]:
display(Markdown('## Placeholder Counts'))
placeholder_rows = pd.DataFrame([
    {'token': key, 'count': value}
    for key, value in summary['placeholder_counts'].items()
])
display(placeholder_rows)

display(Markdown('## Token Counts By Modality / Role'))
display(pd.DataFrame(build_modality_breakdown(analysis)))


In [ ]:
display(Markdown('## What Is Masked vs Supervised For Loss'))
mask_segments = pd.DataFrame(build_mask_segments(analysis, max_segments=16))
display(mask_segments)

display(Markdown('## Decoded Training Target'))
display(Markdown(f"```text\n{analysis['decoded_target']}\n```"))
